In [1]:
!pip -q install dlt[duckdb]

In [2]:
import dlt
from itertools import islice  # islice - Python helper for previewing records
from dlt.sources.rest_api import rest_api_source  #used to define an API source

In [3]:
def openlibrary_source(query: str = "harry potter"):
    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })

In [4]:
pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" # logs the pipeline run (Optiona)
)

In [5]:
extract_info = pipeline.extract(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 184.50 MB (31.40%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.24s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 6875908.20/s
Memory usage: 186.00 MB (31.40%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.40s | Rate: 0.00/s
books: 400  | Time: 1.16s | Rate: 344.63/s
Memory usage: 186.75 MB (31.40%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.85s | Rate: 0.00/s
books: 700  | Time: 2.62s | Rate: 267.63/s
Memory usage: 188.00 MB (31.40%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.85s | Rate: 0.

In [6]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: ['books']
Tables: ['books']
Load ID: 1772252694.6306577

Resource: books
rows extracted: 3759



In [7]:
normalize_info = pipeline.normalize()

------------------- Normalize rest_api in 1772252694.6306577 -------------------
Files: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 196.66 MB (31.70%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772252694.6306577 -------------------
Files: 0/2 (0.0%) | Time: 0.01s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 196.66 MB (31.70%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772252694.6306577 -------------------
Files: 10/2 (500.0%) | Time: 2.83s | Rate: 3.53/s
Items: 0  | Time: 2.82s | Rate: 0.00/s
Memory usage: 201.11 MB (31.70%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772252694.6306577 -------------------
Files: 10/2 (500.0%) | Time: 2.88s | Rate: 3.48/s
Items: 23083  | Time: 2.87s | Rate: 8056.20/s
Memory usage: 201.11 MB (31.70%) | CPU usage: 0.00%



In [8]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")

Load ID: 1772252694.6306577

Tables created/updated:
  - books: 3759 rows
  - books__author_key: 4637 rows
  - books__author_name: 4637 rows
  - books__ia: 3434 rows
  - books__ia_collection: 2735 rows
  - books__language: 3748 rows
  - books__id_standard_ebooks: 12 rows
  - books__id_librivox: 64 rows
  - books__id_project_gutenberg: 56 rows


In [ ]:
# Display schema 
pipeline.default_schema

In [10]:
load_info = pipeline.load()

--------------------- Load rest_api in 1772252694.6306577 ----------------------
Jobs: 0/10 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 225.15 MB (32.40%) | CPU usage: 0.00%

--------------------- Load rest_api in 1772252694.6306577 ----------------------
Jobs: 8/10 (80.0%) | Time: 1.05s | Rate: 7.62/s
Memory usage: 328.96 MB (33.70%) | CPU usage: 0.00%

--------------------- Load rest_api in 1772252694.6306577 ----------------------
Jobs: 10/10 (100.0%) | Time: 1.60s | Rate: 6.24/s
Memory usage: 250.20 MB (32.70%) | CPU usage: 0.00%



In [ ]:
load_info = pipeline.run(openlibrary_source())

In [11]:
ds = pipeline.dataset()

In [12]:
ds.tables

['books',
 'books__author_key',
 'books__author_name',
 'books__ia',
 'books__ia_collection',
 'books__language',
 'books__id_standard_ebooks',
 'books__id_librivox',
 'books__id_project_gutenberg',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [13]:
df = ds.books.df()      # main table
df.head(3)

,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,key,lending_edition_s,lending_identifier_s,public_scan_b,title,_dlt_load_id,_dlt_id,subtitle
0,OL61027601M,15155833,borrowable,397,1997,True,/works/OL82563W,OL38565767M,harrypotterylapi0000rowl_q5r6,False,Harry Potter and the Philosopher's Stone,1772252694.6306577,ITFGKfglNMW/vg,None
1,OL26378158M,15158660,printdisabled,144,2007,True,/works/OL82586W,None,None,False,Harry Potter and the Deathly Hallows,1772252694.6306577,XfxSc+etmJGwBQ,None
2,OL26234270M,10580435,borrowable,279,1999,True,/works/OL82536W,OL48101764M,bdrc-W8LS66814,False,Harry Potter and the Prisoner of Azkaban,1772252694.6306577,8L4FA+VoBFKVfw,None
